# Hoffbauer Antiprimes - Berechnung mit SageMath

Dieses Notebook implementiert die Hoffbauer-Dirac-Operator-Berechnungen für Antiprimes mit SageMath 10.8.

In [ ]:
# SageMath Imports
from sage.all import *
# Sicherstellen, dass CC verfügbar ist (ComplexField für komplexe Zahlen)
if 'CC' not in dir():
    CC = ComplexField(53)  # Standard-Genauigkeit für komplexe Zahlen

# Standard Python-Bibliotheken
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Familien-Definitionen

In [ ]:
# ---------- Familien (deine Definition) ----------
def fam_mod12(p):
    r = int(p % 12)
    if r == 1:  return "e"
    if r == 5:  return "a"
    if r == 7:  return "b"
    if r == 11: return "c"
    return "x"

def fam_phase(p):
    mp = {"e":0.0, "a":np.pi/2, "b":np.pi, "c":3*np.pi/2, "x":0.0}
    return mp[fam_mod12(p)]

In [4]:
## Quadrupel (e, a, b, c) mit Familien E, A, B, C

Berechnung aller Quadrupel, wobei:
- e ∈ Familie E (≡ 1 mod 12)
- a ∈ Familie A (≡ 5 mod 12)
- b ∈ Familie B (≡ 7 mod 12)
- c ∈ Familie C (≡ 11 mod 12)
- n = e² + a² + b² + c² ≤ 1.000.000

SyntaxError: invalid character '∈' (U+2208) (947264607.py, line 4)

In [ ]:
# Funktionen für Quadrupel (e, a, b, c) mit Familien E, A, B, C
# Hinweis: Stellen Sie sicher, dass Zelle 1 (Imports) zuerst ausgeführt wurde!
from sage.all import Integer, is_squarefree, sqrt
from itertools import product

def get_family_from_number(n):
    """Gibt die Familie einer Zahl zurück (e, a, b, c oder None)"""
    r = int(n % 12)
    if r == 1:  return "e"
    if r == 5:  return "a"
    if r == 7:  return "b"
    if r == 11: return "c"
    return None

def get_numbers_by_family(max_value, family):
    """Sammelt alle Zahlen einer Familie bis max_value"""
    numbers = []
    for n in range(1, max_value + 1):
        if get_family_from_number(n) == family:
            numbers.append(n)
    return numbers

def generate_quadruples(max_n=1000000):
    """
    Generiert alle Quadrupel (e, a, b, c) wobei:
    - e in Familie E (== 1 mod 12)
    - a in Familie A (== 5 mod 12)
    - b in Familie B (== 7 mod 12)
    - c in Familie C (== 11 mod 12)
    - n = e^2 + a^2 + b^2 + c^2 <= max_n
    
    Args:
        max_n: Maximale Summe n = e^2 + a^2 + b^2 + c^2 (default: 1.000.000)
    """
    # Bestimme maximale Werte für e, a, b, c
    max_coord = int(sqrt(max_n)) + 1
    
    print(f"Berechne Quadrupel bis n = {max_n:,}")
    print(f"Maximaler Koordinatenwert: {max_coord}")
    print("=" * 60)
    
    # Sammle Zahlen nach Familien
    print("Sammle Zahlen nach Familien...")
    family_e = get_numbers_by_family(max_coord, "e")
    family_a = get_numbers_by_family(max_coord, "a")
    family_b = get_numbers_by_family(max_coord, "b")
    family_c = get_numbers_by_family(max_coord, "c")
    
    print(f"  Familie E: {len(family_e)} Zahlen (z.B. {family_e[:10]}...)")
    print(f"  Familie A: {len(family_a)} Zahlen (z.B. {family_a[:10]}...)")
    print(f"  Familie B: {len(family_b)} Zahlen (z.B. {family_b[:10]}...)")
    print(f"  Familie C: {len(family_c)} Zahlen (z.B. {family_c[:10]}...)")
    
    results = []
    total_checked = 0
    
    print(f"\nGeneriere Quadrupel...")
    
    # Iteriere durch alle möglichen Kombinationen
    for e in family_e:
        e_sq = e * e
        if e_sq > max_n:
            break
        
        for a in family_a:
            a_sq = a * a
            if e_sq + a_sq > max_n:
                break
            
            for b in family_b:
                b_sq = b * b
                if e_sq + a_sq + b_sq > max_n:
                    break
                
                for c in family_c:
                    c_sq = c * c
                    n = e_sq + a_sq + b_sq + c_sq
                    total_checked += 1
                    
                    if n > max_n:
                        break
                    
                    # Prüfe ob n quadratfrei ist
                    if is_squarefree(n):
                        results.append({
                            "n": n,
                            "e": e,
                            "a": a,
                            "b": b,
                            "c": c,
                            "P": e * a * b * c,  # Produkt
                            "L": n,  # Summe der Quadrate
                            "e_family": "E",
                            "a_family": "A",
                            "b_family": "B",
                            "c_family": "C"
                        })
                    
                    # Fortschrittsanzeige
                    if total_checked % 100000 == 0:
                        print(f"  {total_checked:,} Kombinationen geprüft, {len(results):,} gültige Quadrupel gefunden...")
    
    print(f"\n{'=' * 60}")
    print(f"Fertig: {total_checked:,} Kombinationen geprüft")
    print(f"{len(results):,} quadratfreie Quadrupel gefunden")
    print(f"{'=' * 60}")
    
    return results

### Berechnung aller Quadrupel bis 1.000.000

In [ ]:
# Stelle sicher, dass pandas importiert ist
try:
    pd
except NameError:
    print("Hinweis: pandas wird jetzt importiert (Zelle 1 sollte zuerst ausgeführt werden)")
    import pandas as pd

# Berechne alle Quadrupel bis n = 1.000.000
results = generate_quadruples(max_n=1000000)

# Erstelle DataFrame für bessere Übersicht
df_quadruples = pd.DataFrame(results)

# Sortiere nach n
df_quadruples = df_quadruples.sort_values('n').reset_index(drop=True)

print("\nErste 50 Quadrupel:")
print(df_quadruples.head(50).to_string())

Berechne Quadrupel bis n = 1,000,000
Maximaler Koordinatenwert: 1001
Sammle Zahlen nach Familien...
  Familie E: 84 Zahlen (z.B. [1, 13, 25, 37, 49, 61, 73, 85, 97, 109]...)
  Familie A: 84 Zahlen (z.B. [5, 17, 29, 41, 53, 65, 77, 89, 101, 113]...)
  Familie B: 83 Zahlen (z.B. [7, 19, 31, 43, 55, 67, 79, 91, 103, 115]...)
  Familie C: 83 Zahlen (z.B. [11, 23, 35, 47, 59, 71, 83, 95, 107, 119]...)

Generiere Quadrupel...
  100,000 Kombinationen geprüft, 0 gültige Quadrupel gefunden...
  200,000 Kombinationen geprüft, 0 gültige Quadrupel gefunden...
  300,000 Kombinationen geprüft, 0 gültige Quadrupel gefunden...
  400,000 Kombinationen geprüft, 0 gültige Quadrupel gefunden...
  500,000 Kombinationen geprüft, 0 gültige Quadrupel gefunden...
  600,000 Kombinationen geprüft, 0 gültige Quadrupel gefunden...
  700,000 Kombinationen geprüft, 0 gültige Quadrupel gefunden...
  800,000 Kombinationen geprüft, 0 gültige Quadrupel gefunden...
  900,000 Kombinationen geprüft, 0 gültige Quadrupel gef

NameError: name 'pd' is not defined

### Analyse der Quadrupel

In [8]:
# Statistiken
print(f"\nGesamtanzahl Quadrupel: {len(df_quadruples):,}")
print(f"\nKleinste und größte n:")
print(f"  Kleinste: n = {df_quadruples['n'].min()}")
print(f"  Größte: n = {df_quadruples['n'].max()}")

print(f"\nKleinste und größte Produkte P = e*a*b*c:")
print(f"  Kleinste: P = {df_quadruples['P'].min()}")
print(f"  Größte: P = {df_quadruples['P'].max()}")

print(f"\nErste 20 Quadrupel (sortiert nach n):")
for idx, row in df_quadruples.head(20).iterrows():
    print(f"  n={row['n']:6d}: e={row['e']:3d}, a={row['a']:3d}, b={row['b']:3d}, c={row['c']:3d} | P={row['P']:,}")

# Zeige einige interessante Beispiele
print(f"\nBeispiele mit verschiedenen n-Werten:")
examples = df_quadruples.iloc[::max(1, len(df_quadruples)//10)]
for idx, row in examples.head(10).iterrows():
    print(f"  n={row['n']:6d} = {row['e']}² + {row['a']}² + {row['b']}² + {row['c']}² = {row['e']**2} + {row['a']**2} + {row['b']**2} + {row['c']**2}")

NameError: name 'df_quadruples' is not defined

### Visualisierung der Quadrupel

In [ ]:
# Stelle sicher, dass pandas und matplotlib importiert sind
try:
    pd
except NameError:
    import pandas as pd
try:
    plt
except NameError:
    import matplotlib.pyplot as plt

# Visualisierung
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Verteilung von n
axes[0, 0].hist(df_quadruples['n'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('n = e² + a² + b² + c²')
axes[0, 0].set_ylabel('Häufigkeit')
axes[0, 0].set_title('Verteilung von n')
axes[0, 0].grid(True, alpha=0.3)

# 2. Verteilung von P (Produkt)
axes[0, 1].hist(df_quadruples['P'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_xlabel('P = e * a * b * c')
axes[0, 1].set_ylabel('Häufigkeit')
axes[0, 1].set_title('Verteilung von P (Produkt)')
axes[0, 1].grid(True, alpha=0.3)

# 3. n vs P (Scatter)
axes[1, 0].scatter(df_quadruples['n'], df_quadruples['P'], alpha=0.5, s=10)
axes[1, 0].set_xlabel('n')
axes[1, 0].set_ylabel('P')
axes[1, 0].set_title('n vs P')
axes[1, 0].grid(True, alpha=0.3)

# 4. Verteilung der Koordinaten
coords = pd.concat([
    df_quadruples[['e']].rename(columns={'e': 'value'}).assign(coord='e'),
    df_quadruples[['a']].rename(columns={'a': 'value'}).assign(coord='a'),
    df_quadruples[['b']].rename(columns={'b': 'value'}).assign(coord='b'),
    df_quadruples[['c']].rename(columns={'c': 'value'}).assign(coord='c')
])
for coord in ['e', 'a', 'b', 'c']:
    axes[1, 1].hist(df_quadruples[coord], bins=30, alpha=0.5, label=coord.upper())
axes[1, 1].set_xlabel('Koordinatenwert')
axes[1, 1].set_ylabel('Häufigkeit')
axes[1, 1].set_title('Verteilung der Koordinaten e, a, b, c')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Export der Ergebnisse

In [ ]:
# Speichere Ergebnisse in CSV
output_filename = 'quadruple_familien_eabc.csv'
df_quadruples.to_csv(output_filename, index=False)
print(f"✓ Ergebnisse gespeichert in '{output_filename}'")
print(f"  {len(df_quadruples):,} Quadrupel gespeichert")

# Zeige Zusammenfassung
print(f"\nZusammenfassung:")
print(f"  Spalten: n, e, a, b, c, P, L, e_family, a_family, b_family, c_family")
print(f"  Format: n = e^2 + a^2 + b^2 + c^2, wobei e in E, a in A, b in B, c in C")

## Hoffbauer-Hilfsfunktionen

In [ ]:
# ---------- Hoffbauer-Hilfen ----------
def antiprime_from_gap(p_i, p_next):
    g = int(p_next - p_i)
    return int(p_i + g//2), g

def index_window(i, N, w):
    lo = max(0, i - w)
    hi = min(N-1, i + w)
    return range(lo, hi+1)

## Hauptfunktionen: A_theta und Dirac-Operator

In [ ]:
# ---------- A_theta aus Hoffbauer-Schnitt ----------
def A_theta_hoffbauer(theta, N=200, w=3, mu=0.35, kappa=0.55, eta=0.20,
                      use_family_phase=True, normalize=True):
    P = [int(nth_prime(i+1)) for i in range(N+1)]
    primes = P[:N]
    primes_next = P[1:N+1]

    gaps, antip = [], []
    for i in range(N):
        a_i, g_i = antiprime_from_gap(primes[i], primes_next[i])
        antip.append(a_i)
        gaps.append(g_i)

    lg = np.log(np.array(gaps, dtype=float))
    lg = (lg - lg.mean()) / (lg.std() + 1e-12)

    A = zero_matrix(CC, N, N)

    # Diagonal: Gap-Potential
    for i in range(N):
        A[i,i] = CC(mu * lg[i])

    # Kette: gerichteter Fluss (mit Holonomie theta)
    ph_edge = complex(np.cos(theta), np.sin(theta))
    for i in range(N-1):
        w_i = kappa / np.sqrt(gaps[i] + 1e-12)
        if use_family_phase:
            extra = np.exp(1j * (fam_phase(primes[i]) - fam_phase(primes[i+1])))
        else:
            extra = 1.0
        A[i, i+1] += CC(w_i * ph_edge * extra)
        A[i+1, i] += CC(w_i * np.conjugate(ph_edge) * np.conjugate(extra))

    # Hoffbauer-Fensterkopplungen über Antiprime-Nähe
    antip_arr = np.array(antip, dtype=float)
    scale = np.median(np.abs(np.diff(antip_arr))) + 1e-12

    for i in range(N):
        for j in index_window(i, N, w):
            if j == i:
                continue
            dist = abs(antip_arr[i] - antip_arr[j]) / scale
            wij = eta * np.exp(-dist) / np.sqrt((gaps[i]+1e-12)*(gaps[j]+1e-12))
            if use_family_phase:
                extra = np.exp(1j * (fam_phase(primes[i]) - fam_phase(primes[j])))
            else:
                extra = 1.0
            A[i,j] += CC(wij * ph_edge * extra)

    if normalize:
        nr = float(max(np.abs(np.array(A.list(), dtype=complex))))
        if nr > 0:
            A = (1.0/nr) * A
    return A

def make_chiral_dirac(A):
    N = A.nrows()
    Z = zero_matrix(CC, N, N)
    return block_matrix([[Z, A],
                         [A.conjugate_transpose(), Z]])

def det_phase(D, t=0.0, eps=1e-6):
    n = D.nrows()
    I = identity_matrix(CC, n)
    Z = D - t*I + (CC(0, eps))*I
    val = Z.det()
    return np.angle(complex(val))

def winding_number(N=200, thetas=361, t=0.0, eps=1e-6, w=3, params=None):
    th = np.linspace(0.0, 2.0*np.pi, thetas)
    phases = []
    for theta in th:
        A = A_theta_hoffbauer(theta, N=N, w=w, **(params or {}))
        D = make_chiral_dirac(A)
        phases.append(det_phase(D, t=t, eps=eps))
    phases = np.unwrap(np.array(phases))
    W = (phases[-1] - phases[0])/(2.0*np.pi)
    return float(W)

def spectral_flow(N=200, thetas=361, t=0.0, w=3, params=None):
    th = np.linspace(0.0, 2.0*np.pi, thetas)
    evs = []
    for theta in th:
        A = A_theta_hoffbauer(theta, N=N, w=w, **(params or {}))
        D = make_chiral_dirac(A) - t*identity_matrix(CC, 2*N)
        lam = np.array([float(rr) for rr in D.eigenvalues()])
        lam.sort()
        evs.append(lam)
    evs = np.array(evs)

    sf = 0
    for j in range(evs.shape[1]):
        s = np.sign(evs[:, j])
        s[s == 0] = 1
        flips = np.where(s[1:] != s[:-1])[0]
        for k in flips:
            if evs[k, j] < 0 and evs[k+1, j] > 0: sf += 1
            if evs[k, j] > 0 and evs[k+1, j] < 0: sf -= 1
    return int(sf)

## 1) Robustheits-Grid Berechnung

In [ ]:
# ---------- 1) Robustheits-Grid ----------
N0 = 200
thetas = 361

grid_w  = [2,3,4]
grid_eps = [1e-4, 1e-6, 1e-8]

rows = []
for w in grid_w:
    for eps in grid_eps:
        W = winding_number(N=N0, thetas=thetas, t=0.0, eps=eps, w=w)
        SF = spectral_flow(N=N0, thetas=thetas, t=0.0, w=w)
        rows.append({"N":N0, "w":w, "eps":eps, "W(t=0)":W, "SF(t=0)":SF})

df = pd.DataFrame(rows)
print(df)

## 2) "137-Umfeld"-Diagnose

In [ ]:
# ---------- 2) "137-Umfeld"-Diagnose ----------
# Idee: wir betrachten zwei Cutoffs:
#   A) N=200 (enthält p=137 sicher)
#   B) N=200, aber wir "maskieren" die ersten m Knoten (setzen Kanten dort auf 0),
#      um zu sehen, ob die frühen Strukturen (inkl. 137-Region) topologisch dominieren.
def A_theta_masked(theta, N=200, mask_first=0, **kwargs):
    A = A_theta_hoffbauer(theta, N=N, **kwargs)
    if mask_first <= 0:
        return A
    # maskiere Zeilen/Spalten im A-Block -> entfernt frühe Knoten dynamisch
    for i in range(mask_first):
        for j in range(N):
            A[i,j] = 0
            A[j,i] = 0
    return A

def winding_number_masked(mask_first, N=200, w=3, eps=1e-6, thetas=361):
    th = np.linspace(0.0, 2.0*np.pi, thetas)
    phases = []
    for theta in th:
        A = A_theta_masked(theta, N=N, mask_first=mask_first, w=w)
        D = make_chiral_dirac(A)
        phases.append(det_phase(D, t=0.0, eps=eps))
    phases = np.unwrap(np.array(phases))
    return float((phases[-1]-phases[0])/(2*np.pi))

masks = [0, 20, 40, 60, 80]  # grob: "entferne" frühe Bereiche (137 liegt bei Index ~33)
Wmask = [winding_number_masked(m, N=N0, w=3, eps=1e-6, thetas=thetas) for m in masks]

## Visualisierungen

In [ ]:
# Plot: Sensitivität von W gegen frühe Regionen
plt.figure()
plt.plot(masks, Wmask, marker="o")
plt.xlabel("mask_first (Anzahl entfernter früher Knoten)")
plt.ylabel("Winding W bei t=0")
plt.title("Hoffbauer-Dirac: Sensitivität von W gegen frühe Regionen (inkl. 137)")
plt.grid(True)
plt.show()

In [ ]:
# Plot Robustheits-Grid als Heatmap-ähnliche Darstellung
pivot = df.pivot(index="w", columns="eps", values="W(t=0)")
plt.figure()
plt.imshow(pivot.values, aspect="auto")
plt.xticks(range(len(pivot.columns)), [str(e) for e in pivot.columns])
plt.yticks(range(len(pivot.index)), [str(w) for w in pivot.index])
plt.xlabel("eps")
plt.ylabel("w")
plt.title("W(t=0) über (w, eps)")
plt.colorbar()
plt.show()